# Colab + Google Drive Learning Cycle (GPU Template)

**リポジトリ・データがすでに Drive にある前提**です。clone は不要です。

1. Mount Google Drive
2. Drive 上のリポジトリに移動し、依存関係をインストール
3. GPU 確認後、Drive 上の `data/raw` で学習・予測
4. 成果物を同じ Drive ツリーに保存


## 0. Runtime setting

Before running, set Colab runtime to GPU:

- Runtime -> Change runtime type -> Hardware accelerator: **GPU**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# Drive 上のリポジトリルート（要: マウント後のパスに合わせる）
PROJECT_DIR = '/content/drive/MyDrive/app/kyotei_Prediction'

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(f'Drive にリポジトリが見つかりません: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
# requirements-colab: リポジトリ直下 or 親フォルダ(app) を探す
def find_req():
    for name in ('requirements-colab.txt', 'requirements-colab'):
        p = os.path.join(PROJECT_DIR, name)
        if os.path.isfile(p):
            return p
    parent = os.path.dirname(PROJECT_DIR)
    for name in ('requirements-colab.txt', 'requirements-colab'):
        p = os.path.join(parent, name)
        if os.path.isfile(p):
            return p
    return os.path.join(PROJECT_DIR, 'requirements.txt')
req_file = find_req()
print('Using', req_file)
!python -m pip install --upgrade pip -q
!pip install -r {req_file} -q


In [ ]:
# GPU check
!nvidia-smi

import torch
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda device:', torch.cuda.get_device_name(0))


In [ ]:
import os

# Optional CUDA memory behavior tuning for long runs
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# データ・成果物のルート（リポジトリと同じなら PROJECT_DIR のまま）
DRIVE_ROOT = PROJECT_DIR
# 取得データ（リポジトリ内は kyotei_predictor/data/raw）
DATA_DIR = f'{DRIVE_ROOT}/kyotei_predictor/data/raw'
YEAR_MONTH = '2026-02'
PREDICT_DATE = '2026-02-14'

# Presets:
# - quick_check: shortest validation run
# - night_train: longer overnight training run
PROFILES = {
    'quick_check': {
        'N_TRIALS': 1,
        'MINIMAL': True,
        'RUN_FETCH': False,
        'FETCH_START_DATE': '2026-02-01',
        'FETCH_END_DATE': '2026-02-14',
        'RUN_PREDICTION': True,
    },
    'night_train': {
        'N_TRIALS': 20,
        'MINIMAL': False,
        'RUN_FETCH': False,
        'FETCH_START_DATE': '2026-01-01',
        'FETCH_END_DATE': '2026-02-14',
        'RUN_PREDICTION': True,
    },
}

PROFILE_NAME = 'quick_check'  # <- change to 'night_train' for longer run
if PROFILE_NAME not in PROFILES:
    raise ValueError(f'unknown PROFILE_NAME: {PROFILE_NAME}')

profile = PROFILES[PROFILE_NAME]
N_TRIALS = profile['N_TRIALS']
MINIMAL = profile['MINIMAL']
RUN_FETCH = profile['RUN_FETCH']
FETCH_START_DATE = profile['FETCH_START_DATE']
FETCH_END_DATE = profile['FETCH_END_DATE']
RUN_PREDICTION = profile['RUN_PREDICTION']

# Optional manual overrides
# N_TRIALS = 5
# MINIMAL = False

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print('PROFILE_NAME    =', PROFILE_NAME)
print('DRIVE_ROOT      =', DRIVE_ROOT)
print('DATA_DIR        =', DATA_DIR)
print('YEAR_MONTH      =', YEAR_MONTH)
print('PREDICT_DATE    =', PREDICT_DATE)
print('N_TRIALS        =', N_TRIALS)
print('MINIMAL         =', MINIMAL)
print('RUN_FETCH       =', RUN_FETCH)
print('FETCH_START_DATE=', FETCH_START_DATE)
print('FETCH_END_DATE  =', FETCH_END_DATE)
print('RUN_PREDICTION  =', RUN_PREDICTION)


In [ ]:
# 取得データを Google Drive に保存（RUN_FETCH が True のとき実行）
# 保存先: DATA_DIR = /content/drive/MyDrive/kyotei_prediction/data/raw

if RUN_FETCH:
    !python -m kyotei_predictor.tools.batch.batch_fetch_all_venues \
      --start-date "{FETCH_START_DATE}" \
      --end-date "{FETCH_END_DATE}" \
      --stadiums ALL \
      --output-data-dir "{DATA_DIR}"
    print('取得データの保存先（Google Drive）:', DATA_DIR)
    if os.path.exists(DATA_DIR):
        n = len([f for r, d, files in os.walk(DATA_DIR) for f in files])
        print(f'  → 保存ファイル数: {n}')


In [ ]:
import subprocess

cmd = [
    'python',
    'scripts/run_colab_learning_cycle.py',
    '--drive-root', DRIVE_ROOT,
    '--data-dir', DATA_DIR,
    '--year-month', YEAR_MONTH,
    '--n-trials', str(N_TRIALS),
    '--predict-date', PREDICT_DATE,
]
if MINIMAL:
    cmd.append('--minimal')
if not RUN_PREDICTION:
    cmd.append('--skip-prediction')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
import os

# 予測結果はローカル出力後、Drive へ同期済み
pred_path = f'outputs/predictions_{PREDICT_DATE}.json'
drive_pred_path = f'{DRIVE_ROOT}/outputs/predictions_{PREDICT_DATE}.json'
print('prediction file (local):', os.path.exists(pred_path), pred_path)
print('prediction file (Drive):', os.path.exists(drive_pred_path), drive_pred_path)

path = pred_path if os.path.exists(pred_path) else (drive_pred_path if os.path.exists(drive_pred_path) else None)
if path and os.path.exists(path):
    with open(path, 'r', encoding='utf-8') as f:
        pred = json.load(f)
    print('prediction_date:', pred.get('prediction_date'))
    print('summary:', pred.get('execution_summary'))

print('Drive artifact roots（取得データ・学習結果の保存先）:')
for d in ['data/raw', 'optuna_models', 'optuna_results', 'optuna_logs', 'outputs']:
    p = f'{DRIVE_ROOT}/{d}'
    print('-', p, 'exists=' + str(os.path.exists(p)))
